# Training `word2vec` from scratch 

## Step 1: Data pre-processing

In [1]:
import torch 
from torch import nn, optim, tensor, Tensor 
import nltk, re 

In [2]:
FILE_PATH = '/kaggle/input/datasets/luisgasparcordeiro/14-million-word-corpus-txt/corpus.txt'
f = open(FILE_PATH)
corpus = f.read()
f.close()

Now we tokenise the sentences and clean them up. 

In [3]:
raw_sentences = nltk.sent_tokenize(corpus)
tokenized_sentences = []
token_counts = {}
token_set : set[str] = set()

for sentence in raw_sentences: 
    clean = re.sub('[^A-Za-z]', ' ', sentence.lower())
    if len(clean) < 5: continue 
    tokenized = nltk.word_tokenize(clean)
    for token in tokenized: 
        if token in token_set: 
            token_counts[token] += 1
        else:
            token_counts[token] = 1
            token_set.add(token)
    tokenized_sentences.append(tokenized)

## Step 2: Handling tokens 

In [4]:
idx2word = []
word2idx = {}
OOV : set[str] = set()

for token in token_set: 
    if token_counts[token] > 10:
        word2idx[token] = len(idx2word)
        idx2word.append(token)
    else:
        OOV.add(token) 

In [5]:
sentence_tensors : list[Tensor] = []

for sentence in tokenized_sentences: 
    sentence_tensors.append(tensor(
        [word2idx[word] for word in sentence if word not in OOV]
    ))

Note: There are 33121 unique tokens

## Step 3: Preparing the model

In [6]:
counts = tensor([token_counts[word] for word in idx2word])

In [7]:
neg_probs = torch.pow(counts, 3/4)
neg_probs /= neg_probs.sum()

Time to prepare the actual model now!

In [ ]:
class Word2VecTrainer(nn.Module): 
    def __init__(self, vocab_size, embed_size): 
        super().__init__()
        self.in_embed = nn.Embedding(vocab_size, embed_size)
        self.out_embed = nn.Embedding(vocab_size, embed_size)
    
    def forward(self, center : Tensor, word : Tensor, negs : Tensor):
        a : Tensor = self.in_embed(center) # (b, e)
        b : Tensor = self.out_embed(word) # (b, e)
        c : Tensor = self.out_embed(negs) # (b, k, e)
        output_1 = torch.sum(a*b, dim=1)
        output_2 = torch.bmm(a.unsqueeze(dim=1), c.transpose(1, 2)).squeeze(dim=1)
        return output_1, output_2

In [9]:
VOCAB_SIZE = len(idx2word)
EMBED_SIZE = 128 

model = Word2VecTrainer(VOCAB_SIZE, EMBED_SIZE)
model.to('cuda')
loss_fn = nn.BCEWithLogitsLoss()
optimiser = optim.Adam(model.parameters(), lr=1e-4)

## Step 4: Training the model 

In [12]:

BATCH_SIZE = 128

def train_one_epoch(epoch : int, dbg : bool = False): 
    center_batch : list[Tensor] = []
    word_batch : list[Tensor] = []

    cur_size = 0
    print("Epoch:", epoch)
    losses = 0
    tot = 0
    for i in range(500000):
        sentence = sentence_tensors[i] 
        if i % 10000 == 0 and tot > 0 : 
            print('sentences seen:', i)
            print('loss right now:', losses/tot)
            losses = 0
            tot = 0
        if len(sentence) < 5: continue 
        centers = torch.arange(2, len(sentence)-2)
        for d in [-2, -1, +1, +2]: 
            center = sentence[centers]
            word = sentence[centers + d]
            center_batch.append(center)
            word_batch.append(word)
            cur_size += len(center)
        
        if cur_size >= BATCH_SIZE: 
            # feed batch to model 
            if dbg: print("Running batch with cur_size:", cur_size)
            final_center_batch = torch.cat(center_batch).to('cuda')
            final_word_batch = torch.cat(word_batch).to('cuda')
            final_neg_batch = torch.multinomial(neg_probs, len(final_center_batch) * 5, replacement=True).view(len(final_center_batch), 5)
            final_neg_batch = final_neg_batch.to('cuda')

            # train model 
            preds : tuple[Tensor, Tensor] = model(final_center_batch, final_word_batch, final_neg_batch)
            pos, neg = preds 
            pos_loss = loss_fn(pos, torch.ones_like(pos))
            neg_loss = loss_fn(neg, torch.zeros_like(neg)).mean()
            if dbg: print("Loss =", pos_loss.item(), '+', neg_loss.item())
            loss : Tensor = (pos_loss + neg_loss) / 2
            losses += loss.item()
            tot += 1
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
            

            # reset 
            center_batch.clear()
            word_batch.clear()
            cur_size -= cur_size 
    
    print(f"Epoch {epoch} over.")
    print("Loss:", losses/tot)

for epoch in range(5):
    try: 
        train_one_epoch(epoch)
    finally: 
        torch.save(model, '/kaggle/working/model.pth')

Epoch: 0
sentences seen: 10000
loss right now: 3.535700872423676
sentences seen: 20000
loss right now: 3.3800531287878433
sentences seen: 30000
loss right now: 3.2312882796050495
sentences seen: 40000
loss right now: 3.110675345814232
sentences seen: 50000
loss right now: 3.0574617137426374
sentences seen: 60000
loss right now: 3.085352863680189
sentences seen: 70000
loss right now: 2.9830757901943916
sentences seen: 80000
loss right now: 2.8430952741613256
sentences seen: 90000
loss right now: 2.706093320991108
sentences seen: 100000
loss right now: 2.603139857389067
sentences seen: 110000
loss right now: 2.4883358552164014
sentences seen: 120000
loss right now: 2.390527802364082
sentences seen: 130000
loss right now: 2.2973549380146117
sentences seen: 140000
loss right now: 2.233480000438386
sentences seen: 150000
loss right now: 2.1652156473428206
sentences seen: 160000
loss right now: 2.1255282439347067
sentences seen: 170000
loss right now: 2.104694368602001
sentences seen: 180000

Looks like the model is performing and learning well! Let's see how it has performed so far 

In [20]:
torch.save(model.state_dict(), '/kaggle/working/saved_model.pth')

## Step 5: Evaluating the embedding

In [ ]:
embeddings = model.in_embed.to('cpu')

We normaise the embeddings for better performance with cosine similarity. Also we want to detach the vector from autogad

In [45]:
embeddings.weight.detach_()
embeddings.weight /= embeddings.weight.norm(dim=1, keepdim=True)

In [46]:
def find_similar(word, num : int = 5): 
    t0 = tensor(word2idx[word])
    e0 = embeddings(t0)
    sims : list[Tensor] = []
    for i in range(len(idx2word)):
        if idx2word[i] == word: continue 
        sims.append(tensor(i))
    sims.sort(key = lambda t: -torch.dot(e0, embeddings(t)).item())
    return [idx2word[int(t.item())] for t in sims[:num]]
    


In [82]:
def find_similar_tensor(e0, num : int = 5): 
    sims : list[Tensor] = []
    for i in range(len(idx2word)):
        sims.append(tensor(i))
    sims.sort(key = lambda t: -torch.dot(e0, embeddings(t)).item())
    return [idx2word[int(t.item())] for t in sims[:num]]
def embed(word): 
    return embeddings(tensor(word2idx[word]))

Let's see it in action!

In [54]:
find_similar('leader')

['former', 'president', 'head', 'director', 'said']

In [55]:
find_similar('court')

['office', 'federal', 'law', 'meeting', 'government']

In [56]:
find_similar('man')

['who', 'game', 'he', 'when', 'few']

Clearly, the model is learning to group words by context, and we can see similar words being grouped together! But not all words appear frequently enough for the model to have learnt them...

In [57]:
find_similar('king')

['linde', 'onesavings', 'distributed', 'cemig', 'writer']

In [58]:
find_similar('writer')

['inspired', 'king', 'neeraj', 'overbearing', 'disarm']

Let's look at the model's political views and test its understanding of politics and economics, since that seems to be what the training data focuses on. 

In [60]:
find_similar('republican')

['white', 'presidential', 'former', 'president', 'court']

In [222]:
find_similar('democrat')

['told', 'lawmakers', 'investigation', 'clinton', 'trump']

In [177]:
find_similar('trump')

['mr', 'she', 'he', 'congress', 'donald']

In [180]:
find_similar('war')

['members', 'world', 'blocs', 'american', 'friend']

In [67]:
find_similar('election')

['week', 'thursday', 'court', 'meeting', 'trump']

In [69]:
find_similar('law')

['court', 'state', 'federal', 'government', 'office']

In [73]:
find_similar('senate')

['house', 'congress', 'lawmakers', 'trump', 'glancing']

In [74]:
find_similar('congress')

['trump', 'him', 'he', 'mr', 'she']

In [76]:
find_similar('term')

['short', 'long', 'debt', 'or', 'current']

In [77]:
find_similar('dollars')

['pounds', 'moore', 'pursuing', 'euros', 'grow']

In [79]:
find_similar('england')

['uk', 'robert', 'khris', 'london', 'deliveries']

In [88]:
find_similar('american')

['national', 's', 'world', 'including', 'largest']

In [115]:
find_similar('black')

['assemblyman', 'hirst', 'kuala', 'toys', 'handpicked']

In [116]:
find_similar('white')

['republican', 'former', 'said', 'president', 'told']

In [183]:
find_similar('liberal')

['mtv', 'young', 'swath', 'californians', 'samantha']

In [184]:
find_similar('conservative')

['members', 'urged', 'member', 'country', 'budget']

In [ ]:
find_similar('unemployment')

['yen', 'diminished', 'targets', 'eurozone', 'stagnating']

In [187]:
find_similar('inflation')

['growth', 'prices', 'rising', 'rates', 'demand']

In [136]:
find_similar('bill')

['commission', 'plan', 'house', 'sources', 'government']

In [209]:
find_similar('oil')

['prices', 'rose', 'production', 'fell', 'china']